In [2]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
RANDOM_STATE = 42
DATA_PATH = 'marketing_campaign.csv'
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

sns.set_style('whitegrid')


# ===========================================================================
# 1. Load
# ===========================================================================
def load_data(path):
    df = pd.read_csv(path, sep='\t')
    print(f"Loaded: {df.shape}")
    print(f"Missing: {df.isnull().sum()[df.isnull().sum() > 0].to_dict()}")
    return df


# ===========================================================================
# 2. Preprocess + Feature Engineering
# ===========================================================================
def preprocess(df):
    """
    Clean data and add derived features for clustering.
    """
    df = df.copy()

    # Parse date and compute tenure
    df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='%d-%m-%Y')
    ref_date = df['Dt_Customer'].max()
    df['Customer_Tenure_Days'] = (ref_date - df['Dt_Customer']).dt.days

    # Derived features
    df['Age'] = 2014 - df['Year_Birth']
    df['TotalChildren'] = df['Kidhome'] + df['Teenhome']
    df['TotalSpending'] = df[[
        'MntWines', 'MntFruits', 'MntMeatProducts',
        'MntFishProducts', 'MntSweetProducts', 'MntGoldProds'
    ]].sum(axis=1)
    df['TotalPurchases'] = df[[
        'NumDealsPurchases', 'NumWebPurchases',
        'NumCatalogPurchases', 'NumStorePurchases'
    ]].sum(axis=1)
    df['CampaignAcceptedTotal'] = df[[
        'AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3',
        'AcceptedCmp4', 'AcceptedCmp5'
    ]].sum(axis=1)

    # Outliers (business rules)
    n0 = len(df)
    df = df[df['Year_Birth'] >= 1940]
    df = df[(df['Income'].isnull()) | (df['Income'] < 200_000)]
    print(f"Outlier removal: {n0} -> {len(df)} rows")

    # Drop unused columns
    df = df.drop(columns=['Z_CostContact', 'Z_Revenue',
                          'Dt_Customer', 'Year_Birth'])
    return df


# ===========================================================================
# 3. Choose k
# ===========================================================================
def choose_k(X_scaled, k_range=range(2, 9)):
    """
    Evaluate k by inertia (elbow) and silhouette.
    Returns the two arrays for plotting and the silhouette-winning k.
    """
    inertias, silhouettes = [], []
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        labels = km.fit_predict(X_scaled)
        inertias.append(km.inertia_)
        silhouettes.append(silhouette_score(X_scaled, labels))

    silhouette_best = list(k_range)[int(np.argmax(silhouettes))]
    print("Silhouette scores by k:")
    for k, s in zip(k_range, silhouettes):
        print(f"  k={k}: {s:.3f}")
    print(f"Silhouette winner: k={silhouette_best}")
    return inertias, silhouettes, silhouette_best


# ===========================================================================
# 4. Run clustering
# ===========================================================================
def run_clustering(df):
    """
    K-Means customer segmentation with persona labeling.
    """
    # Features for clustering. Response is EXCLUDED here -- used only
    # afterwards to interpret each cluster.
    cluster_features = [
        'Income', 'Age', 'TotalChildren', 'Customer_Tenure_Days', 'Recency',
        'TotalSpending', 'TotalPurchases', 'NumWebVisitsMonth',
        'NumDealsPurchases', 'CampaignAcceptedTotal',
    ]
    print(f"\nClustering features ({len(cluster_features)}): "
          f"{cluster_features}")

    X = df[cluster_features].copy()
    X = X.fillna(X.median())

    # Scaling is required for K-Means (Euclidean distance)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # --- Choose k ---
    k_range = range(2, 9)
    inertias, silhouettes, silhouette_best = choose_k(X_scaled, k_range)

    # We intentionally choose k=4 rather than the silhouette winner (k=2).
    # k=2 only splits high vs. low income — too coarse for marketing personas.
    # k=4 gives 4 distinguishable personas with clear strategy implications.
    final_k = 4
    print(f"\nFinal choice: k={final_k} "
          f"(silhouette winner is k={silhouette_best}, "
          f"but k=4 gives richer personas)")

    km = KMeans(n_clusters=final_k, random_state=RANDOM_STATE, n_init=10)
    df = df.copy()
    df['Cluster'] = km.fit_predict(X_scaled)

    # --- Cluster profiles ---
    profile_cols = ['Income', 'Age', 'TotalChildren', 'TotalSpending',
                    'TotalPurchases', 'NumWebVisitsMonth',
                    'NumDealsPurchases', 'CampaignAcceptedTotal', 'Recency']
    profile = df.groupby('Cluster')[profile_cols].mean().round(1)
    profile['Cluster_Size'] = df.groupby('Cluster').size()
    profile['Response_Rate'] = df.groupby('Cluster')['Response'].mean().round(3)
    profile = profile.sort_values('Response_Rate', ascending=False)
    print(f"\nCluster profiles (k={final_k}):")
    print(profile.to_string())

    # --- Auto-generate persona labels ---
    overall_income_med = df['Income'].median()
    overall_spend_med = df['TotalSpending'].median()
    print("\nPersonas (auto-labeled):")
    for c in profile.index:
        row = profile.loc[c]
        income_lvl = "High" if row['Income'] > overall_income_med else "Low"
        spend_lvl = "High" if row['TotalSpending'] > overall_spend_med else "Low"
        resp_lvl = ("Responsive" if row['Response_Rate'] > 0.15
                    else "Unresponsive")
        print(f"  Cluster {c}: {income_lvl}-Income, {spend_lvl}-Spend, "
              f"{resp_lvl}  (n={row['Cluster_Size']}, "
              f"resp={row['Response_Rate']:.1%})")

    # --- Plots: elbow + silhouette + PCA + response by cluster ---
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].plot(list(k_range), inertias, 'o-', linewidth=2)
    axes[0, 0].axvline(final_k, ls='--', color='red', alpha=0.5,
                       label=f'Chosen k={final_k}')
    axes[0, 0].set_title('Elbow Method')
    axes[0, 0].set_xlabel('Number of clusters (k)')
    axes[0, 0].set_ylabel('Inertia')
    axes[0, 0].legend()

    axes[0, 1].plot(list(k_range), silhouettes, 'o-',
                    color='orange', linewidth=2)
    axes[0, 1].axvline(final_k, ls='--', color='red', alpha=0.5,
                       label=f'Chosen k={final_k}')
    axes[0, 1].set_title('Silhouette Score')
    axes[0, 1].set_xlabel('Number of clusters (k)')
    axes[0, 1].set_ylabel('Silhouette')
    axes[0, 1].legend()

    # PCA projection
    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    coords = pca.fit_transform(X_scaled)
    colors = plt.cm.Set2(np.linspace(0, 1, final_k))
    for i, c in enumerate(sorted(df['Cluster'].unique())):
        mask = df['Cluster'] == c
        axes[1, 0].scatter(coords[mask, 0], coords[mask, 1],
                           label=f'Cluster {c} (n={mask.sum()})',
                           alpha=0.6, s=20, color=colors[i])
    axes[1, 0].set_title(f'Clusters in PCA Space '
                         f'(explained var: '
                         f'{pca.explained_variance_ratio_.sum():.1%})')
    axes[1, 0].set_xlabel('PC1')
    axes[1, 0].set_ylabel('PC2')
    axes[1, 0].legend(fontsize=8)

    # Response rate by cluster
    resp_by_cluster = df.groupby('Cluster')['Response'].mean().sort_values()
    bars = axes[1, 1].bar(resp_by_cluster.index.astype(str),
                          resp_by_cluster.values,
                          color=[colors[c] for c in resp_by_cluster.index])
    axes[1, 1].axhline(df['Response'].mean(), ls='--', color='red',
                       label=f'Overall mean ({df["Response"].mean():.1%})')
    axes[1, 1].set_title('Campaign Response Rate by Cluster')
    axes[1, 1].set_xlabel('Cluster')
    axes[1, 1].set_ylabel('Response Rate')
    axes[1, 1].legend()
    for bar, val in zip(bars, resp_by_cluster.values):
        axes[1, 1].text(bar.get_x() + bar.get_width()/2, val + 0.005,
                        f'{val:.1%}', ha='center', fontsize=9)

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/clustering_results.png',
                dpi=120, bbox_inches='tight')
    plt.close()
    print(f"\nSaved: {OUTPUT_DIR}/clustering_results.png")

    # --- Cluster comparison heatmap (z-score normalized) ---
    profile_numeric = profile[profile_cols + ['Response_Rate']]
    profile_z = (profile_numeric - profile_numeric.mean()) / profile_numeric.std()
    fig, ax = plt.subplots(figsize=(11, 5))
    sns.heatmap(profile_z, annot=profile_numeric.round(1), fmt='g',
                cmap='RdBu_r', center=0, ax=ax,
                cbar_kws={'label': 'Z-score'})
    ax.set_title('Cluster Profile Comparison '
                 '(color = z-score, value = mean)')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/cluster_heatmap.png',
                dpi=120, bbox_inches='tight')
    plt.close()
    print(f"Saved: {OUTPUT_DIR}/cluster_heatmap.png")

    profile.to_csv(f'{OUTPUT_DIR}/cluster_profiles.csv')
    print(f"Saved: {OUTPUT_DIR}/cluster_profiles.csv")


# ===========================================================================
# Main
# ===========================================================================
def main():
    print("=" * 60)
    print("Customer Personality Analysis - Clustering")
    print("=" * 60)
    df = load_data(DATA_PATH)
    df = preprocess(df)
    run_clustering(df)
    print("\nDone.")


if __name__ == '__main__':
    main()


Customer Personality Analysis - Clustering
Loaded: (2240, 29)
Missing: {'Income': 24}
Outlier removal: 2240 -> 2236 rows

Clustering features (10): ['Income', 'Age', 'TotalChildren', 'Customer_Tenure_Days', 'Recency', 'TotalSpending', 'TotalPurchases', 'NumWebVisitsMonth', 'NumDealsPurchases', 'CampaignAcceptedTotal']
Silhouette scores by k:
  k=2: 0.249
  k=3: 0.216
  k=4: 0.215
  k=5: 0.161
  k=6: 0.156
  k=7: 0.149
  k=8: 0.147
Silhouette winner: k=2

Final choice: k=4 (silhouette winner is k=2, but k=4 gives richer personas)

Cluster profiles (k=4):
          Income   Age  TotalChildren  TotalSpending  TotalPurchases  NumWebVisitsMonth  NumDealsPurchases  CampaignAcceptedTotal  Recency  Cluster_Size  Response_Rate
Cluster                                                                                                                                                                
3        80749.1  42.4            0.2         1663.3            21.1                3.4                1